# Representations

How each cluster becomes a **single profile**, set with `ClusterConfig(representation=...)`.

| representation | the profile is… | preserves |
|----------------|-----------------|-----------|
| `medoid` | a real period from the cluster | realistic shape |
| `mean` | the average of the cluster | central tendency |
| `maxoid` | the most extreme period | peaks |
| `distribution` | reshaped to match the value histogram | the **duration curve** |
| `distribution_minmax` | distribution + exact per-step min/max | distribution **and** extremes |
| `minmax_mean` | mean, but min/max kept per timestep | extremes around the average |

**There is no single default** — it follows the clustering method:

| method | default representation |
|---|---|
| `hierarchical` (default), `kmedoids`, `contiguous` | `medoid` |
| `kmeans`, `averaging` | `mean` |
| `kmaxoids` | `maxoid` |

So switching to `kmeans` silently switches you to `mean` as well. Setting `representation=`
explicitly overrides that for any method — the two levers are independent.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## They score differently on different things

**RMSE** measures point-by-point timing; **RMSE on the duration curve** measures how well the
*spread* of values is kept. A representation can win one and lose the other:

In [ ]:
from tsam import ClusterConfig

reps = [
    "medoid",
    "mean",
    "maxoid",
    "distribution",
    "distribution_minmax",
    "minmax_mean",
]
rows = {}
for rep in reps:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    rows[rep] = {
        "RMSE": round(float(r.accuracy.rmse.mean()), 4),
        "RMSE (duration curve)": round(float(r.accuracy.rmse_duration.mean()), 4),
    }
pd.DataFrame(rows).T

## On the duration curve

Sorting every value high-to-low ignores *when* things happen and shows the distribution. `mean`
and `medoid` shave the peaks and fill the valleys; `distribution` is built to trace this curve:

In [ ]:
import plotly.express as px

frames = []
for rep in ["medoid", "mean", "distribution", "minmax_mean"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    s = r.reconstructed["Load"].sort_values(ascending=False).reset_index(drop=True)
    frames.append(
        pd.DataFrame({"rank": range(len(s)), "Load": s.values, "representation": rep})
    )
orig = data["Load"].sort_values(ascending=False).reset_index(drop=True)
frames.append(
    pd.DataFrame(
        {"rank": range(len(orig)), "Load": orig.values, "representation": "original"}
    )
)
px.line(
    pd.concat(frames),
    x="rank",
    y="Load",
    color="representation",
    title="Duration curve by representation",
)

## What `mean` clips

`mean` quietly loses the maximum; `distribution` keeps it:

In [ ]:
bounds = {"original": [data["Load"].max(), data["Load"].min()]}
for rep in ["mean", "distribution"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    bounds[rep] = [r.reconstructed["Load"].max(), r.reconstructed["Load"].min()]
pd.DataFrame(bounds, index=["peak Load", "min Load"]).round(1)

The same choice applies one level down: `SegmentConfig(representation=...)` takes exactly these
options for [segments](segmentation.ipynb).

---

* [Extreme periods](extreme_periods.ipynb) — force one *specific* period to survive verbatim.
* [Clustering methods](clustering_methods.ipynb) — the companion lever.
* [Comparing representations](../tutorials/comparing_representations.ipynb) — all six on one
  cluster, scored.